In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append(str(Path("../src").resolve()))
from quantlab.data.binance import load_parquet

In [ ]:
df = load_parquet("../data/raw/BTCUSDT_1d.parquet")

In [ ]:
# Features

df['returns'] = (df['close'] - df['close'].shift(1)) / df['close'].shift(1)
df['log_returns'] = np.log(1 + df['returns'])
df['rolling_vol_30'] = df['log_returns'].rolling(30).std()
df['sma_20'] = df['close'].rolling(20).mean()
df['sma_50'] = df['close'].rolling(50).mean()
df['momentum_20'] = df['close']/df['close'].shift(20) -1

In [ ]:
df[['returns','log_returns', 'rolling_vol_30', 'sma_20', 'sma_50', 'momentum_20']].isna().sum()

In [ ]:
df[['close', 'sma_20', 'sma_50']].plot(figsize = (12, 6), grid = True)

In [ ]:
df[['log_returns', 'rolling_vol_30', 'momentum_20']].describe().T

In [ ]:
df[['returns', 'log_returns', 'rolling_vol_30', 'momentum_20']].hist(figsize=(12, 8))

Let's understand if the features have some predictive power, e.g., by comparing the future prediction with momentum:

In [ ]:
df['future_return_1d'] = df['returns'].shift(-1) # this is not a feature, this is a prediction target!

In [ ]:
momementum_20_fut_ret_corr = df[['momentum_20','future_return_1d']].corr().iloc[0,1]
print('Correlation between momementum_20 and future returns is {}'.format(round(momementum_20_fut_ret_corr,4)))

Another parameter to check is how much the price gets far from a recent trend (i.e., a moving average): if this parameter is positive and increasing, then the price tends to get bigger and bigger wrt to the trend, so, if the parameter is alse predictive, then there is some chance for a momentum strategy.

In [ ]:
df['dist_price_sma50'] = df['close'] / df['sma_50'] - 1
price_50_d_trend_dist_corr = df[['dist_price_sma50','future_return_1d']].corr().iloc[0,1]
print('Correlation between price-to-50_days_trend distance and future returns is {}'.format(round(price_50_d_trend_dist_corr,4)))

A stochastic process $X_t$ is defined **strongly stationary** if every finite dimensional distribution is time invariant, i.e., if
$$
(X_{t_1},\dots,X_{t_n})\stackrel{\text d}{=}(X_{t_1+h},\dots,X_{t_n+h})
$$
while it is **weakly stationary** if
- $\Bbb E[X_t]$ and $\operatorname{Var}[X_t]$ are constant
- $\operatorname{Cov}[X_{t+h},X_t]=\gamma(h)$ depends only on the lag $h$.
  
In practice we check whether mean and variance are constant in time, taking 30-days time windows and measuring its std. The covariance check is done on a larger window (260 days) to make this measure less noisy.

Stationarity measures how stable the statistical properties of a time series are.

We say that $X_t$ shows **autocorrelation** when
$$
\operatorname{Corr}[X_{t+h},X_t]\neq0
$$
Autocorrelation measures whether past values contain linear information about future values.

Are returns stationary or autocorrelated?

In [ ]:
rolling_mean = df['returns'].rolling(30).mean().rename('rolling_mean')
rolling_var = df['returns'].rolling(30).var().rename('rolling_var')
rolling_cov = [df['returns'].rolling(260).cov(df['returns'].shift(h)).rename('rolling_cov with shift {}'.format(h)) for h in range(1, 21)]

ts = [
    rolling_mean,
    rolling_var,
    rolling_cov[0],
    rolling_cov[4],
]

In [ ]:
fig, axes = plt.subplots(len(ts), figsize = (10, 3 * len(ts)))

for ax, series in zip(axes, ts):
    series.plot(ax = ax,
               label = 'std = {}'.format(round(series.std(), 5)))
    ax.set_title(series.name)
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()

If $X_t\sim\mathcal N(\mu,\sigma^2)$ are iid, then $\hat\rho(h)\approx\mathcal N(0,1/n)$. Since for normal standard $Z$ one has $\textbf P(|Z|<2)\approx 95\%$, it makes sense to compare the sample autocorrelations at different lags with the approximate 95% confidence band $2/\sqrt n$.

In [ ]:
len_ret = df['returns'].dropna().shape[0]
threshold =  2 / (len_ret)**(0.5)
lags = range(1,21)
auto_corrs = [df['returns'].autocorr(lag=h) for h in lags]

plt.figure(figsize = (10,5))
plt.scatter(lags, auto_corrs)
plt.axhline(threshold, color='red', linestyle = '--', label='$\pm 2/\sqrt{n}$')
plt.axhline(-threshold, color='red', linestyle = '--')
plt.axhline(0, color='black', linestyle = '-', linewidth=0.3)
plt.title('Returns autocorrelation with different lags')
plt.xlabel('lags')
plt.ylabel('autocorrelartion')


plt.legend()
plt.show()